In [1]:
cd ..

/home/bhchen/LearnKalmanGain


In [2]:
# test_ns_gen_data.py
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import animation

from utils import gen_data


def main():
    """
    End-to-end test for NS using gen_data().

    This script:
      1) Generates trajectories via gen_data
      2) Reshapes (T, N*N) -> (T, N, N)
      3) Saves an animated GIF for quick visual inspection
    """

    dataset = "ns"

    # Time grid (only dt matters)
    T = 1000
    dt = 0.1
    t = torch.linspace(0, T * dt, T + 1)

    # gen_data params
    steps_test = 10
    steps_valid = 5

    class Args:
        ori_dim = None

    args = Args()

    print("Generating NS data using gen_data...")

    v_traj, v_valid, v_test = gen_data(
        dataset=dataset,
        t=t,
        steps_test=steps_test,
        steps_valid=steps_valid,
        args=args,
        v0=None,
        sigma_v=0,
        check_disk=False,
        steps_burn=1000,
        dt_iter=10,
        prefix="debug_",
        test_only=False,
    )

    print("Train traj shape:", v_traj.shape)
    print("Valid traj shape:", v_valid.shape)
    print("Test traj shape:", v_test.shape)

    # (T+1, B, N*N)
    dim = v_traj.shape[2]
    N = int(np.sqrt(dim))
    print(f"Inferred grid size: N={N}, batch={v_traj.shape[1]}")

    # Convert to numpy: (T+1, N, N) for the first batch member
    data = v_traj[:, 0].view(-1, N, N).cpu().numpy()

    # Basic sanity stats (use nan-safe ops just in case)
    print("Sanity check on amplitude:")
    print("max:", np.nanmax(data))
    print("min:", np.nanmin(data))
    print("std:", np.nanstd(data))

    # -----------------------------
    # GIF configuration
    # -----------------------------
    gif_path = "ns_etdrk4_gen_data.gif"

    # To keep the GIF small and fast, subsample frames
    # Example: take every 20th frame
    frame_stride = 20
    frames = data[::frame_stride]
    times = (np.arange(frames.shape[0]) * frame_stride) * dt

    # Use a fixed color range for stable visuals across time
    vmin = np.nanpercentile(frames, 1)
    vmax = np.nanpercentile(frames, 99)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = -1.0, 1.0

    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(
        frames[0],
        origin="lower",
        extent=[0, 2 * np.pi, 0, 2 * np.pi],
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
        animated=True,
    )
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    title = ax.set_title(f"t={times[0]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    def update(i):
        im.set_array(frames[i])
        title.set_text(f"t={times[i]:.2f}")
        return (im, title)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=50,   # milliseconds between frames
        blit=False,
    )

    # Save GIF (requires pillow)
    # pip install pillow
    writer = animation.PillowWriter(fps=20)
    ani.save(gif_path, writer=writer, dpi=120)
    plt.close(fig)

    print(f"Saved GIF to: {gif_path}")


if __name__ == "__main__":
    main()




Generating NS data using gen_data...
Train traj shape: torch.Size([1002, 1, 4096])
Valid traj shape: torch.Size([5, 1, 4096])
Test traj shape: torch.Size([11, 1, 4096])
Inferred grid size: N=64, batch=1
Sanity check on amplitude:
max: 38.607468
min: -38.535374
std: 5.339713
Saved GIF to: ns_etdrk4_gen_data.gif
